In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer


In [19]:
movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)
movies.head()


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [17]:
ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="latin-1"
)
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [22]:
#multi_hot_encoding genres
movies['genres_list'] = movies['genres'].str.split('|')

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genres_list'])

genre_df = pd.DataFrame(
    genre_matrix,
    columns=mlb.classes_,
    index=movies['movie_id']
)
genre_df

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movie_id,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3948,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3949,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3950,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [7]:
#content_based_similarity(movie X movie)
genre_similarity = cosine_similarity(genre_df)


In [23]:
user_item_matrix = ratings.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating',
    fill_value=0
)
user_item_matrix

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,0.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
#collaborative_filtering_similarity(movie X user)
cf_similarity = cosine_similarity(user_item_matrix.T)


In [10]:
common_movie_ids = user_item_matrix.columns.intersection(movies['movie_id'])

genre_sim_aligned = pd.DataFrame(
    genre_similarity,
    index=movies['movie_id'],
    columns=movies['movie_id']
).loc[common_movie_ids, common_movie_ids]

cf_sim_aligned = pd.DataFrame(
    cf_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
).loc[common_movie_ids, common_movie_ids]


In [11]:
alpha = 0.5  # balance between content and CF

hybrid_similarity = (
    alpha * genre_sim_aligned +
    (1 - alpha) * cf_sim_aligned
)
#if alpha is low(0.0,0.1,..) it gives importance to cf and viseversa

In [12]:
def recommend_hybrid(movie_title, movies, hybrid_similarity, top_n=5):

    if movie_title not in movies['title'].values:
        return "Movie not found."

    movie_id = movies[movies['title'] == movie_title]['movie_id'].values[0]

    if movie_id not in hybrid_similarity.index:
        return "Movie not available for hybrid recommendation."

    sim_scores = hybrid_similarity.loc[movie_id]
    sim_scores = sim_scores.sort_values(ascending=False)

    recommended_ids = sim_scores.index[1:top_n + 1]

    return movies[movies['movie_id'].isin(recommended_ids)][['title', 'genres']]


In [13]:
recommend_hybrid(
    "Toy Story (1995)",
    movies,
    hybrid_similarity,
    top_n=5
)


,title,genres
584,Aladdin (1992),Animation|Children's|Comedy|Musical
2072,"American Tail, An (1986)",Animation|Children's|Comedy
2286,"Bug's Life, A (1998)",Animation|Children's|Comedy
3045,Toy Story 2 (1999),Animation|Children's|Comedy
3682,Chicken Run (2000),Animation|Children's|Comedy


In [ ]:
### Insight
'''
A weighted hybrid recommender was implemented by combining genre-based content similarity with collaborative filtering derived from real user ratings.
By weighting both similarity sources, the system balances personalization with content relevance and mitigates cold-start limitations 
inherent in individual recommendation approaches. '''
